In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Activation, BatchNormalization, Dropout
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import BinaryAccuracy, Precision, Recall, AUC
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve
import joblib, gc

FEATURES_SELECCIONADAS = [
    'iat', 'rst_count', 'urg_count', 'number', 'variance', 'tot_size',
    'max', 'header_length', 'flow_duration', 'weight', 'rate', 'duration',
    'protocol_type', 'syn_flag_number', 'fin_count', 'syn_count',
    'rst_flag_number', 'ack_count'
]
NOMBRE_CLASE_BENIGNA = 'BenignTraffic'

print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
df = pd.read_feather('df_binary_balanced.feather')
df['label_binario'] = (df['label'] != NOMBRE_CLASE_BENIGNA).astype(int)
print('Shape:', df.shape)
print(df['label_binario'].value_counts())


In [ ]:
X = df[FEATURES_SELECCIONADAS].values
y = df['label_binario'].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=2/9, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

del df; gc.collect()

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}')
print(f'Features: {X_train.shape[1]}')


In [ ]:
def focal_loss(gamma=2.0, alpha=0.75):
    """
    Focal Loss binaria.
    gamma > 0 enfoca en ejemplos dificiles.
    alpha = peso clase positiva (malicioso).
    NO combinar con class_weight.
    """
    def _loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce = -y_true * tf.math.log(y_pred) - (1.0 - y_true) * tf.math.log(1.0 - y_pred)
        p_t = y_true * y_pred + (1.0 - y_true) * (1.0 - y_pred)
        alpha_t = y_true * alpha + (1.0 - y_true) * (1.0 - alpha)
        return tf.reduce_mean(alpha_t * tf.pow(1.0 - p_t, gamma) * bce)
    return _loss

# FIX 1 — Instanciar focal_loss UNA SOLA VEZ y reutilizar en todo el notebook.
# focal_loss() es un closure que retorna la funcion interna _loss.
# Reutilizar la misma instancia evita ambiguedad al cargar el modelo y compilar.
loss_fn = focal_loss(gamma=2.0, alpha=0.75)


#### Poda de Magnitud (Weight Pruning)
Se utiliza TensorFlow Model Optimization Toolkit para podar gradualmente los pesos menos importantes durante un reentrenamiento corto

In [ ]:
import tensorflow_model_optimization as tfmot

# 1. Cargar el modelo pre-entrenado
#
# FIX 1 (aplicado): custom_objects mapea el nombre interno '_loss' a la instancia
# ya creada. Pasar focal_loss(2, 0.75) directamente crea un objeto nuevo que Keras
# no puede reconciliar con la configuracion serializada del modelo guardado.
base_model = tf.keras.models.load_model(
    'mlp_binario_v2_exp2_focalloss.h5',
    custom_objects={'_loss': loss_fn}   # <-- FIX: era {'_loss': focal_loss(2, 0.75)}
)

# 2. Parametros de poda (60% de escasez final)
pruning_params = {
    'pruning_schedule': tfmot.sparsity.keras.PolynomialDecay(
        initial_sparsity=0.0,
        final_sparsity=0.60,
        begin_step=0,
        end_step=int(163155)  # (1383725 / 128) * 15 epocas
    )
}

# 3. Aplicar wrapper de poda
pruned_model = tfmot.sparsity.keras.prune_low_magnitude(base_model, **pruning_params)

# 4. Compilar con la misma instancia de loss_fn
pruned_model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss=loss_fn,                          # <-- FIX: era focal_loss(2,0.75)
    metrics=[tf.keras.metrics.AUC()]
)

# 5. Fine-tuning de poda
pruned_model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=128,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    validation_data=(X_val, y_val)
)

# 6. Remover wrappers antes de QAT
stripped_pruned_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

print('\nSparsity por capa:')
for layer in stripped_pruned_model.layers:
    if hasattr(layer, 'kernel'):
        w = layer.kernel.numpy()
        print(f'  {layer.name}: {np.mean(w == 0):.2%}')


#### Quantization-Aware Training (QAT) sobre el modelo podado
Tomamos el modelo podado y limitamos sus pesos y activaciones usando qkeras

**Nota sobre arquitectura con capas separadas:**
El modelo usa `BatchNormalization` + `Activation` como capas independientes.
`model_quantize` mapea por nombre de clase Python: `Dense` -> `QDense`, `Activation` -> activacion cuantizada.
`BatchNormalization` se omite del config_dict intencionalmente (se fusiona en sintesis HLS/FPGA).

In [ ]:
# Inspeccion previa: verificar nombres de clase exactos para el config_dict
print('Capas del modelo podado (stripped):')
for i, layer in enumerate(stripped_pruned_model.layers):
    print(f'  [{i:02d}] {type(layer).__name__:30s}  name={layer.name}')


In [ ]:
from qkeras.utils import model_quantize
from qkeras import quantized_bits, quantized_relu

# 1. Configuracion de cuantizacion
#
# FIX 3: El config_dict usaba 'QDense' como clave (nombre del tipo destino)
# cuando debe ser 'Dense' (nombre del tipo origen en el modelo a convertir).
# Ademas, 'Activation' requiere la clave con el nombre de la activacion ('relu'),
# no 'default', para que coincida con Activation('relu') en la arquitectura.
#
# quantized_bits(8, 0, alpha='auto'):
#   8 bits | 0 bits enteros fijos | escala aprendida automaticamente
# quantized_relu(4, 2):
#   4 bits | 2 bits enteros -> rango [0, 3.75)
#   Aumentar el segundo argumento si las activaciones superan ese rango.

config_dict = {
    "Dense": {                                      # <-- FIX: era "QDense"
        "kernel_quantizer": "quantized_bits(8,0,alpha=auto)",
        "bias_quantizer":   "quantized_bits(8,0,alpha=auto)"
    },
    "Activation": {
        "relu": "quantized_relu(4,2)"               # <-- FIX: era {"default": "quantized_relu(4,2)"}
    }
}

# 2. Convertir a modelo QKeras (transfer_weights conserva los pesos podados)
qat_model = model_quantize(
    stripped_pruned_model,
    config_dict,
    activation_bits=8,
    transfer_weights=True
)

print('Modelo QAT construido:')
qat_model.summary()


In [ ]:
# 3. Compilar y reentrenar con QAT
#
# FIX 2 (aplicado): loss=loss_fn en lugar de loss=focal_loss.
# Pasar focal_loss sin parentesis es un error silencioso: Keras lo llama como
# focal_loss(y_true, y_pred), lo que retorna otra funcion en lugar de un escalar,
# causando un fallo en el calculo del gradiente.
qat_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss=loss_fn,                          # <-- FIX: era loss=focal_loss
    metrics=[tf.keras.metrics.AUC()]
)

history_qat = qat_model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_val, y_val),
    callbacks=[
        EarlyStopping(
            monitor='val_auc', patience=5,
            restore_best_weights=True, mode='max'
        )
    ]
)


In [ ]:
# 4. Evaluacion comparativa: base vs. pruned+QAT
for nombre, modelo in [('Base (sin comprimir)', base_model),
                        ('Pruned + QAT',         qat_model)]:
    probs = modelo.predict(X_test, batch_size=128).ravel()
    preds = (probs > 0.5).astype(int)
    print(f'\n=== {nombre} ===')
    print(classification_report(y_test, preds,
          target_names=['Legitimo (0)', 'Malicioso (1)'], digits=4))


In [ ]:
# 5. Guardar modelo final para hls4ml
qat_model.save('modelo_pruned_qat.h5')

with open('modelo_pruned_qat_arch.json', 'w') as f:
    f.write(qat_model.to_json())
qat_model.save_weights('modelo_pruned_qat_weights.h5')

print('Guardado: modelo_pruned_qat.h5')
print('Guardado: modelo_pruned_qat_arch.json')
print('Guardado: modelo_pruned_qat_weights.h5')
